# Handling Large Data: Chunking, PyArrow, and Dask
### ISA 383: Python for Business Analytics

**Learning Objectives:**

By the end of this notebook, you will be able to:

1. Explain why and when pandas becomes a bottleneck for large datasets.
2. Optimize memory usage through dtype selection, category conversion, and downcasting.
3. Process files that do not fit in memory using chunked reading.
4. Use the PyArrow backend for improved performance and native missing value support.
5. Use Dask as a drop-in pandas replacement for larger-than-memory workflows.
6. Choose the right tool based on data size and task complexity.

**Colab setup:**

The next cell installs the two optional packages used in this notebook and creates a temporary working folder automatically. No Conda environment or path selection is required.


## Using this notebook in Google Colab

1. Before editing, select **File > Save a copy in Drive**.
2. Run the cells in order. Some practice cells intentionally wait for your input.
3. The notebook creates any course folders and teaching files it needs automatically.
4. Files under `/content` are temporary and disappear when the Colab runtime resets.
5. Never paste an API key into a notebook cell. Use the **Secrets** panel when instructed.

You do not need to find, copy, or type a file path for the prepared course data.


## Setup

In [ ]:
%pip install -q pyarrow "dask[dataframe]"

from pathlib import Path

DATA_DIR = Path("/content/isa383/11_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
CSV_FILE = str(DATA_DIR / "large_transactions.csv")
PARQUET_FILE = str(DATA_DIR / "large_transactions.parquet")

print("Working folder prepared.")


In [11]:
import pandas as pd
import numpy as np
import time
import os

pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 120)

## Generating a Large Dataset

We will generate a synthetic dataset with 1 million rows to make the performance differences visible. 
This simulates a transaction log with mixed types: strings, integers, floats, and dates.

In [ ]:
# Generate a 1M-row dataset and save to CSV
np.random.seed(42)
n = 1_000_000

large_df = pd.DataFrame({
    'transaction_id': np.arange(1, n + 1),
    'date': np.random.choice(pd.date_range('2020-01-01', '2024-12-31').strftime('%Y-%m-%d'), n),
    'customer_id': np.random.randint(1, 50001, n),
    'product': np.random.choice(['Laptop', 'Phone', 'Tablet', 'Monitor', 'Keyboard', 'Mouse', 'Charger', 'Cable'], n),
    'category': np.random.choice(['Electronics', 'Accessories', 'Computing'], n),
    'region': np.random.choice(['Dubai', 'Abu Dhabi', 'Sharjah', 'Ajman', 'Fujairah', 'RAK', 'UAQ'], n),
    'quantity': np.random.randint(1, 50, n),
    'unit_price': np.round(np.random.uniform(10, 5000, n), 2),
    'discount_pct': np.random.choice([0, 5, 10, 15, 20, np.nan], n),
    'salesperson': np.random.choice([f'SP{i:03d}' for i in range(1, 201)], n),
})
large_df['total'] = (large_df['quantity'] * large_df['unit_price']).round(2)

# Save to CSV
large_df.to_csv(CSV_FILE, index=False)
file_size_mb = os.path.getsize(CSV_FILE) / (1024 ** 2)
print(f"File size: {file_size_mb:.1f} MB")
print(f"Shape: {large_df.shape}")
large_df.head()


---

# Part 1: When Pandas Breaks

Pandas normally loads the entire dataset into memory. A CSV can require several times its on-disk size because text must be parsed into in-memory dtypes, string columns and indexes add overhead, and transformations may create intermediate copies. The exact ratio depends heavily on the data types, so measure it instead of assuming a fixed multiplier.

Let us measure how much memory our 1M-row dataset actually uses.

In [ ]:
# Read the CSV and check memory
start = time.time()
df = pd.read_csv(CSV_FILE)
read_time = time.time() - start

# memory_usage(deep=True) gives the actual memory including string contents
mem_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"Read time: {read_time:.2f}s")
print(f"Memory usage: {mem_mb:.1f} MB")
print(f"File on disk: {file_size_mb:.1f} MB")
print(f"Memory / disk ratio: {mem_mb / file_size_mb:.1f}x")
print()
df.info(memory_usage='deep')


Notice the `object` dtype columns (product, category, region, salesperson). Each string value is stored as a separate Python object on the heap. This is the single biggest source of memory waste in pandas.

---

# Part 2: Memory Optimization in Pandas

Before reaching for a different library, optimize what you have. Three techniques cover most cases: category conversion for strings, numeric downcasting, and specifying dtypes at read time.

## 2.1 Category Dtype for Strings

When a string column has a small number of unique values relative to the total rows (low cardinality), converting it to `category` dtype is the single most impactful optimization. Instead of storing each string repeatedly, pandas stores it once in a lookup table and uses integer codes internally.

In [14]:
# Check cardinality of string columns
for col in df.select_dtypes(include='object').columns:
    print(f"{col:15s} -> {df[col].nunique():>6,} unique values out of {len(df):>10,} rows")

date            ->  1,827 unique values out of  1,000,000 rows
product         ->      8 unique values out of  1,000,000 rows
category        ->      3 unique values out of  1,000,000 rows
region          ->      7 unique values out of  1,000,000 rows
salesperson     ->    200 unique values out of  1,000,000 rows


/var/folders/g2/t63zx_lj13v7t7tqtw3kr72c0000gn/T/ipykernel_10359/1050430171.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns:


In [15]:
# Convert low-cardinality columns to category
df_opt = df.copy()

cat_cols = ['product', 'category', 'region', 'salesperson']
for col in cat_cols:
    before = df_opt[col].memory_usage(deep=True)
    df_opt[col] = df_opt[col].astype('category')
    after = df_opt[col].memory_usage(deep=True)
    print(f"{col:15s}: {before / 1e6:.1f} MB -> {after / 1e6:.1f} MB  ({(1 - after/before)*100:.0f}% reduction)")

product        : 14.1 MB -> 1.0 MB  (93% reduction)
category       : 18.3 MB -> 1.0 MB  (95% reduction)
region         : 13.7 MB -> 1.0 MB  (93% reduction)
salesperson    : 13.0 MB -> 2.0 MB  (85% reduction)


## 2.2 Numeric Downcasting

By default, integers use `int64` (8 bytes) and floats use `float64` (8 bytes). If the actual values fit in a 
smaller type, you can downcast.

In [16]:
# Downcast numeric columns
for col in ['transaction_id', 'customer_id', 'quantity']:
    before = df_opt[col].memory_usage(deep=True)
    df_opt[col] = pd.to_numeric(df_opt[col], downcast='integer')
    after = df_opt[col].memory_usage(deep=True)
    print(f"{col:15s}: {df_opt[col].dtype} -> {before / 1e6:.1f} MB to {after / 1e6:.1f} MB")

for col in ['unit_price', 'total']:
    before = df_opt[col].memory_usage(deep=True)
    df_opt[col] = pd.to_numeric(df_opt[col], downcast='float')
    after = df_opt[col].memory_usage(deep=True)
    print(f"{col:15s}: {df_opt[col].dtype} -> {before / 1e6:.1f} MB to {after / 1e6:.1f} MB")

transaction_id : int32 -> 8.0 MB to 4.0 MB
customer_id    : int32 -> 8.0 MB to 4.0 MB
quantity       : int8 -> 8.0 MB to 1.0 MB
unit_price     : float32 -> 8.0 MB to 4.0 MB
total          : float64 -> 8.0 MB to 8.0 MB


In [17]:
# Total memory comparison
mem_before = df.memory_usage(deep=True).sum() / (1024 ** 2)
mem_after = df_opt.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"Before optimization: {mem_before:.1f} MB")
print(f"After optimization:  {mem_after:.1f} MB")
print(f"Reduction: {(1 - mem_after/mem_before)*100:.0f}%")

Before optimization: 119.4 MB
After optimization:  49.6 MB
Reduction: 58%


## 2.3 Specifying Dtypes at Read Time

You can apply these optimizations at read time so the full-size DataFrame never exists in memory.

In [ ]:
# Read with optimized types from the start
start = time.time()
df_smart = pd.read_csv(
    CSV_FILE,
    dtype={
        'product': 'category',
        'category': 'category',
        'region': 'category',
        'salesperson': 'category',
        'quantity': 'int16',
    },
    parse_dates=['date'],
    usecols=['date', 'product', 'category', 'region', 'quantity', 'unit_price', 'total']
)
read_time_smart = time.time() - start
mem_smart = df_smart.memory_usage(deep=True).sum() / (1024 ** 2)

print(f"Read time: {read_time_smart:.2f}s (vs {read_time:.2f}s naive)")
print(f"Memory: {mem_smart:.1f} MB (vs {mem_mb:.1f} MB naive)")


## 2.4 Parquet: A Better File Format

CSV is a text format. Every number is stored as characters, type information is not preserved, and reading requires parsing the text. Parquet is a binary, columnar format that stores type metadata and commonly uses compression, so it is often smaller and faster for analytical workloads. If you control the file format, prefer Parquet for typed tabular analytics; keep CSV when human readability or broad interchange is the priority.

In [ ]:
# Save as Parquet
df.to_parquet(PARQUET_FILE, index=False)
parquet_size = os.path.getsize(PARQUET_FILE) / (1024 ** 2)
print(f"CSV size:     {file_size_mb:.1f} MB")
print(f"Parquet size: {parquet_size:.1f} MB")
print(f"Compression:  {(1 - parquet_size/file_size_mb)*100:.0f}% smaller")
print()

# Read speed comparison
start = time.time()
df_csv = pd.read_csv(CSV_FILE)
csv_time = time.time() - start

start = time.time()
df_pq = pd.read_parquet(PARQUET_FILE)
pq_time = time.time() - start

print(f"CSV read:     {csv_time:.2f}s")
print(f"Parquet read: {pq_time:.2f}s")
print(f"Speedup:      {csv_time / pq_time:.1f}x")


Parquet also preserves data types. When you save a DataFrame with category columns and datetime columns to Parquet and read it back, the types are retained. With CSV, everything reverts to strings and integers, and you have to re-specify dtypes on every read.

---

# Part 3: Chunked Processing

When a file is too large to fit in memory even after optimization, you can process it in chunks. The `chunksize` 
parameter in `read_csv` returns an iterator instead of a DataFrame. Each iteration gives you a DataFrame with 
that many rows, which you process and discard before loading the next.

In [ ]:
# chunksize returns an iterator, not a DataFrame
reader = pd.read_csv(CSV_FILE, chunksize=200_000)
print(type(reader))  # TextFileReader, not DataFrame


In [ ]:
# Pattern: accumulate partial results from each chunk, then combine
results = []

start = time.time()
for i, chunk in enumerate(pd.read_csv(CSV_FILE, chunksize=200_000)):
    # Compute grouped sum for this chunk
    partial = chunk.groupby('region')['total'].sum()
    results.append(partial)
    print(f"Chunk {i}: {len(chunk)} rows processed")

# Combine partial results
# Each partial is a Series indexed by region; we sum across all chunks
final = pd.concat(results).groupby(level=0).sum()
elapsed = time.time() - start

print(f"\nTotal time: {elapsed:.2f}s")
print("\nRevenue by region:")
print(final.sort_values(ascending=False))


In [13]:
# Verify against the full DataFrame
expected = df.groupby('region')['total'].sum()
print("Results match:", np.allclose(final.sort_index(), expected.sort_index()))

Results match: True


### What Works Well with Chunking

Chunking works when the operation is decomposable: you can compute a partial result per chunk and combine them. 
Sums, counts, and means (sum + count, then divide) decompose cleanly. Filtering and writing subsets also works: read a chunk, filter it, append to an output file.

### What Does Not Work

Operations that need the full dataset at once do not decompose easily. Sorting the entire dataset, computing a global median, or deduplicating across chunks all require seeing every row. For these, you need Dask.

---

# Part 4: The PyArrow Backend

Since pandas 2.0, you can use Apache Arrow as the memory backend instead of NumPy. Arrow is a columnar 
in-memory format designed for analytical workloads. The key benefits are:

- **Native missing value support**: integer columns with NaN stay as integers instead of being silently upcast to float64. 
In classic pandas, `pd.Series([1, 2, None])` becomes `float64`. With Arrow, it stays `int64` with a proper null.
- **Faster string operations**: Arrow stores strings in a contiguous buffer, not as scattered Python objects.
- **Lower memory usage**: especially for string-heavy datasets (often 40-60% reduction).
- **Interoperability**: Arrow is the lingua franca of modern data tools (Spark, Polars, DuckDB, BigQuery). Data moves between them without copying.

In [ ]:
# Read with the default NumPy backend
start = time.time()
df_numpy = pd.read_csv(CSV_FILE)
numpy_time = time.time() - start
numpy_mem = df_numpy.memory_usage(deep=True).sum() / (1024 ** 2)

print(f"NumPy backend: {numpy_time:.2f}s, {numpy_mem:.1f} MB")
print(df_numpy.dtypes)
print()


In [ ]:
# Read with the PyArrow backend
start = time.time()
df_arrow = pd.read_csv(CSV_FILE, dtype_backend='pyarrow', engine='pyarrow')
arrow_time = time.time() - start
arrow_mem = df_arrow.memory_usage(deep=True).sum() / (1024 ** 2)

print(f"Arrow backend: {arrow_time:.2f}s, {arrow_mem:.1f} MB")
print(df_arrow.dtypes)


In [25]:
# Side-by-side comparison
print(f"{'':20s} {'NumPy':>10s} {'Arrow':>10s} {'Improvement':>12s}")
print("-" * 55)
print(f"{'Read time':20s} {numpy_time:>9.2f}s {arrow_time:>9.2f}s {numpy_time/arrow_time:>10.1f}x faster")
print(f"{'Memory':20s} {numpy_mem:>8.1f} MB {arrow_mem:>8.1f} MB {(1-arrow_mem/numpy_mem)*100:>9.0f}% less")

                          NumPy      Arrow  Improvement
-------------------------------------------------------
Read time                 0.53s      0.70s        0.8x faster
Memory                  119.4 MB     90.9 MB        24% less


## The NaN Problem That Arrow Solves

In classic pandas, a column of integers with a missing value becomes float64 because NumPy integers cannot represent NaN. This is one of the most surprising behaviors for new users.

In [26]:
# Classic pandas: integers with NaN become float
s_numpy = pd.Series([1, 2, None, 4])
print(f"NumPy backend: {s_numpy.dtype}")  # float64
print(s_numpy)
print()

# Arrow backend: integers stay as integers, NaN is a proper null
s_arrow = pd.Series([1, 2, None, 4], dtype='int64[pyarrow]')
print(f"Arrow backend: {s_arrow.dtype}")  # int64[pyarrow]
print(s_arrow)

NumPy backend: float64
0    1.0
1    2.0
2    NaN
3    4.0
dtype: float64

Arrow backend: int64[pyarrow]
0       1
1       2
2    <NA>
3       4
dtype: int64[pyarrow]


## String Performance

Arrow strings are stored as contiguous byte buffers with offsets, rather than as an array of pointers to scattered Python objects. This makes string operations faster and uses less memory.

In [27]:
# String operation speed comparison
test_col_numpy = df_numpy['product']
test_col_arrow = df_arrow['product']

start = time.time()
_ = test_col_numpy.str.lower()
numpy_str_time = time.time() - start

start = time.time()
_ = test_col_arrow.str.lower()
arrow_str_time = time.time() - start

print(f"str.lower() on 1M strings:")
print(f"  NumPy: {numpy_str_time:.3f}s")
print(f"  Arrow: {arrow_str_time:.3f}s")
print(f"  Speedup: {numpy_str_time / arrow_str_time:.1f}x")

str.lower() on 1M strings:
  NumPy: 0.026s
  Arrow: 0.017s
  Speedup: 1.5x


## Current Limitations

The Arrow backend is still maturing. Most common operations work, but you may occasionally hit an edge case where a specific method is not yet optimized or raises an unexpected error. When that happens, you can convert back selectively: `df['col'] = df['col'].astype('object')`. For new projects where you control the stack, Arrow is the right default. For legacy code, test before switching.

### Practice 1: PyArrow Comparison

Read `large_transactions.parquet` twice: once with the default backend and once with `dtype_backend='pyarrow'`. 
Compare read times and memory usage. Is the difference as large as with CSV? Explain why or why not.

In [ ]:
# TODO: Read the Parquet file with both backends, then compare time and memory usage.

---

# Part 5: Introduction to Dask

Dask is a parallel computing library that mirrors the pandas API. You write code that looks almost identical to 
pandas, but Dask splits the data into partitions and processes them in parallel across CPU cores. The key concept 
is **lazy evaluation**: when you call `ddf.groupby('region')['total'].mean()`, Dask does not compute anything. 
It records the operation in a task graph. Computation happens only when you call `.compute()`.

In [32]:
import dask.dataframe as dd

In [ ]:
# Read with Dask - this is instant because nothing is loaded yet
start = time.time()
ddf = dd.read_csv(CSV_FILE)
read_time_dask = time.time() - start

print(f"'Read' time: {read_time_dask:.4f}s  (nothing was actually read)")
print(f"Type: {type(ddf)}")
print(f"Partitions: {ddf.npartitions}")
print()
print("Column types (inferred from first partition):")
print(ddf.dtypes)


Dask split the file into partitions automatically. Each partition is a regular pandas DataFrame. Operations 
are applied to each partition independently where possible.

In [34]:
# Operations are lazy - they build a task graph
result = ddf.groupby('region')['total'].mean()
print(type(result))  # Still a Dask object, not computed
print(result)         # Shows the task graph description

<class 'dask.dataframe.dask_expr._collection.Series'>
Dask Series Structure:
npartitions=1
    float64
        ...
Dask Name: getitem, 5 expressions
Expr=((ArrowStringConversion(frame=FromMapProjectable(71288f3))[['region', 'total']]).mean(observed=True, chunk_kwargs={'numeric_only': False}, aggregate_kwargs={'numeric_only': False}, _slice='total'))['total']


In [35]:
# .compute() triggers execution
start = time.time()
result_computed = result.compute()
elapsed = time.time() - start

print(f"Compute time: {elapsed:.2f}s")
print()
print(result_computed.sort_values(ascending=False))

Compute time: 2.02s

region
Ajman        62804.818964
Sharjah      62764.403657
UAQ          62735.805828
Abu Dhabi    62720.615000
Fujairah     62720.466953
RAK          62572.225719
Dubai        62465.245308
Name: total, dtype: float64


In [36]:
# Most pandas operations work the same way

# Filtering
dubai = ddf[ddf['region'] == 'Dubai']
print(f"Dubai transactions: {len(dubai)}")
print()

# Groupby with named aggregation
summary = ddf.groupby('product').agg(
    total_revenue=('total', 'sum'),
    avg_quantity=('quantity', 'mean'),
    n_transactions=('transaction_id', 'count')
).compute()
print(summary.sort_values('total_revenue', ascending=False))

Dubai transactions: 142683

          total_revenue  avg_quantity  n_transactions
product                                              
Charger    7.910940e+09     25.082827          125357
Laptop     7.845133e+09     24.998952          125017
Tablet     7.842371e+09     25.008471          125128
Monitor    7.839544e+09     25.037428          125146
Keyboard   7.818133e+09     24.989989          124767
Phone      7.816766e+09     24.968577          125005
Cable      7.806915e+09     25.005537          124446
Mouse      7.803525e+09     24.985655          125134


In [40]:
# Method chaining works too
top_products = (
    ddf
    .query('region == "Dubai"')
    .groupby('product')['total']
    .sum()
    .compute()
    .sort_values(ascending=False)
)
print(top_products)

product
Cable       1.121863e+09
Tablet      1.121848e+09
Phone       1.118845e+09
Laptop      1.117828e+09
Charger     1.116515e+09
Mouse       1.112718e+09
Keyboard    1.102861e+09
Monitor     1.100250e+09
Name: total, dtype: float64


In [47]:
(ddf.query('region == "Dubai"')
    .groupby(['product', 'region'])['total'].sum()
    .reset_index(drop = False)
    .sort_values(by = 'total')
    # .compute()
)

,product,region,total
npartitions=1,,,
,string,string,float64
,...,...,...


## What Dask Cannot Do

Dask does not support every pandas operation. The main limitations:

- **No `iloc`**: positional indexing requires knowing the global row order, which does not exist across partitions.
- **Global sorting is expensive**: `sort_values` is supported, but it requires shuffling data across partitions and may be slow.
- **Some methods are missing**: not every pandas method has a Dask equivalent. When you hit one, you can call `.compute()` to get a pandas DataFrame and use the method there.
- **No in-place operations**: Dask DataFrames are immutable task graphs. `inplace=True` does not work.

In [27]:
# Example: iloc does not work
try:
    ddf.iloc[0]
except NotImplementedError as e:
    print(f"Error: {e}")

Error: 'DataFrame.iloc' only supports selecting columns. It must be used like 'df.iloc[:, column_indexer]'.


## When to Use Dask

Dask is useful when your data does not fit in memory and the analysis requires more than simple aggregations. 
If you only need a grouped sum, chunked pandas is simpler. If you need to filter, merge, and aggregate across 
a large dataset, Dask handles the complexity of partitioned computation for you.

For multi-machine workflows, both Dask and PySpark may be appropriate; the choice depends on the existing platform, workload, and team expertise. Dask offers a pandas-like path from a single machine to a cluster, while PySpark is common in established Spark environments.

### Practice 2: Dask

Using Dask, compute the total revenue per region per product (a two-level groupby) from the CSV file. Then find the top 5 region-product combinations by revenue.

In [ ]:
# TODO: Use Dask to compute revenue per region and product, then show the top five combinations.

---

# Part 6: Choosing the Right Tool

| Scenario | Tool | Why |
|---|---|---|
| Data fits comfortably in memory | pandas (default) | Simplest, full API |
| Data fits but is slow or memory-tight | pandas + dtype optimization + PyArrow backend | Same API, better performance |
| Need to process a large file for simple aggregates | Chunked `read_csv` | No new dependencies |
| Data does not fit in memory, complex workflow | Dask | Pandas-like API, handles partitioning |
| Existing multi-node Spark platform | PySpark | Integrates with the Spark ecosystem |

For typed analytical workloads, prefer **Parquet** when you control the file format. Use CSV when human readability or broad interchange matters more.

In [ ]:
# Final comparison: all approaches on the same task
# Task: compute average total per region

print(f"{'Approach':35s} {'Time':>8s} {'Memory':>10s}")
print("-" * 55)

# 1. Naive pandas from CSV
start = time.time()
r1 = pd.read_csv(CSV_FILE).groupby('region')['total'].mean()
t1 = time.time() - start
print(f"{'Pandas (CSV, no optimization)':35s} {t1:>7.2f}s {mem_mb:>8.1f} MB")

# 2. Pandas from Parquet
start = time.time()
r2 = pd.read_parquet(PARQUET_FILE).groupby('region')['total'].mean()
t2 = time.time() - start
mem2 = pd.read_parquet(PARQUET_FILE).memory_usage(deep=True).sum() / (1024**2)
print(f"{'Pandas (Parquet)':35s} {t2:>7.2f}s {mem2:>8.1f} MB")

# 3. Pandas with PyArrow backend
start = time.time()
r3 = pd.read_csv(CSV_FILE, dtype_backend='pyarrow', engine='pyarrow').groupby('region')['total'].mean()
t3 = time.time() - start
print(f"{'Pandas (CSV + Arrow backend)':35s} {t3:>7.2f}s {arrow_mem:>8.1f} MB")

# 4. Chunked pandas
start = time.time()
sums, counts = [], []
for chunk in pd.read_csv(CSV_FILE, chunksize=200_000):
    sums.append(chunk.groupby('region')['total'].sum())
    counts.append(chunk.groupby('region')['total'].count())
r4 = pd.concat(sums).groupby(level=0).sum() / pd.concat(counts).groupby(level=0).sum()
t4 = time.time() - start
print(f"{'Chunked pandas (200K per chunk)':35s} {t4:>7.2f}s {'  low':>10s}")

# 5. Dask
start = time.time()
r5 = dd.read_csv(CSV_FILE).groupby('region')['total'].mean().compute()
t5 = time.time() - start
print(f"{'Dask':35s} {t5:>7.2f}s {'  low':>10s}")


---

# Comprehensive Exercises

### Exercise 1: Optimize and Compare

Write a function `read_optimized(filepath)` that reads a CSV file with the best possible memory optimization 
(category dtypes for low-cardinality string columns, downcast for numerics, date parsing). Test it on 
`large_transactions.csv` and report the memory before and after optimization.

Then save the optimized DataFrame to Parquet, read it back, and verify the dtypes are preserved.

In [ ]:
# Your solution for Exercise 1


### Exercise 2: Chunked Median

Computing a global median from chunks is not straightforward because medians do not decompose the way sums do. 
You cannot combine chunk medians to get the global median.

One approach: read in chunks, collect only the column of interest, and compute the median at the end. But this 
defeats the purpose if the column itself does not fit in memory.

A practical approximation: use chunked reading to build a histogram (using `np.histogram` with fixed bins), 
then estimate the median from the cumulative histogram.

Implement both approaches for the `unit_price` column and compare the results.

*Hint for the histogram approach: create fixed bins (e.g., 1000 bins from min to max), accumulate counts from each chunk, 
then find the bin where the cumulative count crosses 50%.*

In [ ]:
# Your solution for Exercise 2


### Exercise 3: Dask vs Pandas Consistency Check

For each of the following operations, compute the result using both pandas and Dask, then verify they match:

1. Total revenue per product, sorted descending.
2. Number of transactions per region where quantity > 20.
3. The 95th percentile of `total` per category.

For (3), you may discover that Dask handles percentiles differently from pandas. Document what happens.

In [ ]:
# Your solution for Exercise 3


### Exercise 4: The Parquet Column Selection Advantage

Parquet is a columnar format, which means you can read specific columns without loading the entire file. CSV 
requires scanning every row even if you only need one column.

Measure the time to read just the `region` and `total` columns from CSV (using `usecols`) vs from Parquet 
(using `columns`). Compute the ratio. Then measure reading all columns. Does the advantage change?

*Hint: `pd.read_parquet('file.parquet', columns=['region', 'total'])`*

In [ ]:
# Your solution for Exercise 4


### Exercise 5: Building a Large-Data Pipeline

Build a complete analysis pipeline using Dask that:

1. Reads `large_transactions.csv`.
2. Filters to transactions in 2023 and 2024 (parse the date column first).
3. Adds a `year` column.
4. Computes total revenue per region per year.
5. Computes year-over-year growth per region.
6. Saves the final result to a Parquet file.

Do as much as possible in Dask before calling `.compute()`. The goal is to minimize how much data is materialized in memory.

*Hint: Dask supports `assign` for adding columns and `pivot_table` for reshaping. You may need to `.compute()` before 
the year-over-year calculation since that requires comparing rows.*

In [ ]:
# Your solution for Exercise 5


---

## Summary

**Memory optimization** is the first line of defense: convert low-cardinality strings to `category`, downcast 
numerics, specify dtypes at read time, and use `usecols` to load only what you need. These alone can reduce 
memory by 50-80%.

**Parquet** is usually the better choice for typed analytical data because it is commonly smaller, faster, and preserves types; CSV remains useful for readable interchange.

**Chunked processing** handles files that do not fit in memory for decomposable operations like sums and counts. 
It requires no new libraries but demands careful thinking about how to combine partial results.

**PyArrow backend** provides faster reads, lower memory, native null support, and better string performance. 
Enable it with `dtype_backend='pyarrow'` when reading, or use Arrow dtypes directly.

**Dask** mirrors the pandas API for larger-than-memory workflows. It uses lazy evaluation and partitioned 
computation. Use it when chunked processing is too manual and PySpark is too heavy.

The right tool depends on the size of your data and the complexity of your task. Start simple (optimized pandas), 
escalate only when needed.